## Project Overview

This notebook trains and selects a binary classifier for Type 2 Diabetes detection from tongue images.

- **Dataset root**: `/kaggle/input/datasets/edamhakim/tonguepreped/datapreped`
- **Structure**: `train/`, `val/`, `test/` with class folders `diabetes`, `nondiabetes`
- **Leakage rule**: validation and test use only `__orig__` files (all `__aug_` files are excluded)
- **Goal**: train 3 backbones, rank by validation AUC, tune threshold, evaluate on test, and export explainability artifacts

### End-to-End Pipeline (text)

1. Load split folders with a custom dataset class.
2. Apply leakage guard by excluding `__aug_` files from validation and test.
3. Use balanced sampling (`WeightedRandomSampler`) for train batches.
4. Train 3 pretrained backbones with the same two-phase fine-tuning protocol.
5. Save each model checkpoint by best validation AUC.
6. Compare models on validation originals and select the winner.
7. Tune classification threshold on validation (macro F1).
8. Evaluate once on test originals with the chosen threshold.
9. Export plots, GradCAM examples, and `submission.csv`.

## 1. Configuration & Paths

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms
from torchvision import models as tv_models
from torchvision.models import ResNet50_Weights, MobileNet_V3_Large_Weights

import timm

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


@dataclass
class CFG:
    # Paths
    data_root: str = '/kaggle/input/datasets/edamhakim/tonguepreped/datapreped'
    train_dir: str = '/kaggle/input/datasets/edamhakim/tonguepreped/datapreped/train'
    val_dir: str = '/kaggle/input/datasets/edamhakim/tonguepreped/datapreped/val'
    test_dir: str = '/kaggle/input/datasets/edamhakim/tonguepreped/datapreped/test'
    external_dir: str = '/kaggle/input/datasets/edamhakim/tongueexternal/externaldataset'
    output_dir: str = '/kaggle/working'

    # Labels
    class_to_label: Dict[str, int] = None

    # Data
    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 2
    pin_memory: bool = True

    # Train strategy
    warmup_epochs: int = 5
    finetune_epochs: int = 25
    head_lr: float = 1e-3
    ft_lr: float = 1e-4
    eta_min: float = 1e-6
    weight_decay: float = 1e-4
    early_stopping_patience: int = 7

    # MixUp
    mixup_alpha: float = 0.2

    # Repro
    seed: int = 42


cfg = CFG()
cfg.class_to_label = {'diabetes': 1, 'nondiabetes': 0}


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Data root:', cfg.data_root)

## 2. Dataset & DataLoaders

In [ ]:
class TongueDataset(Dataset):
    def __init__(self, root_dir: str, class_to_label: Dict[str, int], transform=None, split_name='train'):
        self.root_dir = Path(root_dir)
        self.class_to_label = class_to_label
        self.transform = transform
        self.split_name = split_name
        self.samples: List[Tuple[Path, int]] = []

        if not self.root_dir.exists():
            raise FileNotFoundError(f'Split path not found: {self.root_dir}')

        for class_name, label in class_to_label.items():
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue

            for p in sorted(class_dir.glob('*.jpg')):
                name = p.name.lower()
                if split_name in ('val', 'test') and '__aug_' in name:
                    # Leakage guard for evaluation splits
                    continue
                self.samples.append((p, label))

        if len(self.samples) == 0:
            raise RuntimeError(f'No samples found in {self.root_dir} for split={split_name}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32), str(path)


train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = TongueDataset(cfg.train_dir, cfg.class_to_label, transform=train_tfms, split_name='train')
val_ds = TongueDataset(cfg.val_dir, cfg.class_to_label, transform=val_test_tfms, split_name='val')
test_ds = TongueDataset(cfg.test_dir, cfg.class_to_label, transform=val_test_tfms, split_name='test')


def count_labels(dataset: TongueDataset):
    labels = [int(y) for _, y in dataset.samples]
    return {0: labels.count(0), 1: labels.count(1)}


def print_split_stats(name: str, ds: TongueDataset):
    counts = count_labels(ds)
    print(f'{name}: total={len(ds)} | nondiabetes(0)={counts[0]} | diabetes(1)={counts[1]}')


print_split_stats('train (orig + aug)', train_ds)
print_split_stats('val (orig only)', val_ds)
print_split_stats('test (orig only)', test_ds)

# WeightedRandomSampler (train only)
train_labels = [label for _, label in train_ds.samples]
class_count = np.bincount(train_labels)
class_weights = 1.0 / np.maximum(class_count, 1)
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    sampler=sampler,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
)

# External Dataset
external_ds = TongueDataset(cfg.external_dir, cfg.class_to_label, transform=val_test_tfms, split_name='external')
print_split_stats('external (full)', external_ds)
external_loader = DataLoader(
    external_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
)

### Why These Three Backbones?

We intentionally compare one strong primary architecture, one classic baseline, and one lightweight model to balance performance, robustness, and efficiency.

- **EfficientNet-B2 (primary candidate)**
  - Good sample efficiency on small medical-image datasets.
  - Compound scaling captures texture and color cues effectively.
  - Often strong accuracy per parameter.

- **ResNet-50 (baseline / sanity benchmark)**
  - Very stable and widely validated architecture.
  - Useful as a reproducible baseline for comparison.
  - Residual design offers different feature behavior from EfficientNet.

- **MobileNetV3-Large (efficient candidate)**
  - Lower compute and memory footprint.
  - Useful for Kaggle runtime constraints and faster experimentation.
  - Good candidate if deployment efficiency matters later.

Using the same data pipeline and training protocol for all three models keeps model selection fair and metric-driven.

## 3. Model Definitions

In [ ]:
def build_model(name: str) -> nn.Module:
    if name == 'efficientnet_b2':
        model = timm.create_model('efficientnet_b2', pretrained=True)
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1408, 1),
        )
        return model

    if name == 'resnet50':
        model = tv_models.resnet50(weights=ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 1),
        )
        return model

    if name == 'mobilenet_v3_large':
        model = tv_models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[3] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 1),
        )
        return model

    raise ValueError(f'Unsupported model name: {name}')


def freeze_all_backbone(model: nn.Module, name: str):
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze head only
    if name == 'efficientnet_b2':
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif name == 'resnet50':
        for p in model.fc.parameters():
            p.requires_grad = True
    elif name == 'mobilenet_v3_large':
        for p in model.classifier.parameters():
            p.requires_grad = True


def partial_unfreeze(model: nn.Module, name: str):
    # Keep everything frozen, then unfreeze specific blocks + head
    freeze_all_backbone(model, name)

    if name == 'efficientnet_b2':
        for p in model.blocks[-2:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

    elif name == 'resnet50':
        for p in model.layer4.parameters():
            p.requires_grad = True
        for p in model.fc.parameters():
            p.requires_grad = True

    elif name == 'mobilenet_v3_large':
        for p in model.features[-3:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True


def get_trainable_params(model: nn.Module):
    return [p for p in model.parameters() if p.requires_grad]


def get_gradcam_target_layer(model: nn.Module, name: str):
    if name == 'efficientnet_b2':
        return [model.blocks[-1]]
    if name == 'resnet50':
        return [model.layer4[-1]]
    if name == 'mobilenet_v3_large':
        return [model.features[-1]]
    raise ValueError(f'Unsupported model name: {name}')


model_names = ['efficientnet_b2', 'resnet50', 'mobilenet_v3_large']

### Training Protocol Notes

- **Phase 1 (warmup)**: train head only to stabilize adaptation to tongue-domain data.
- **Phase 2 (fine-tuning)**: unfreeze selected late blocks and apply lower LR with cosine decay.
- **MixUp** is enabled only in Phase 2 to regularize without hurting initial head convergence.
- **Early stopping** monitors validation AUC to prevent overfitting.

This keeps optimization stable while preserving generalization on a small, medically sensitive dataset.

## 4. Training Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def evaluate(model, loader, device):
    model.eval()
    losses, probs, labels, paths = [], [], [], []

    with torch.no_grad():
        for x, y, p in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with autocast(enabled=(device.type == 'cuda')):
                logits = model(x).squeeze(1)
                loss = criterion(logits, y)

            pred_prob = torch.sigmoid(logits).detach().cpu().numpy()
            losses.append(loss.item())
            probs.extend(pred_prob.tolist())
            labels.extend(y.detach().cpu().numpy().tolist())
            paths.extend(p)

    mean_loss = float(np.mean(losses)) if losses else 0.0
    labels_arr = np.array(labels)
    probs_arr = np.array(probs)
    auc = roc_auc_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.5

    return {
        'loss': mean_loss,
        'auc': float(auc),
        'labels': labels_arr,
        'probs': probs_arr,
        'paths': paths,
    }


def train_one_model(model_name: str):
    print(f'\n===== Training {model_name} =====')
    model = build_model(model_name).to(device)

    freeze_all_backbone(model, model_name)
    optimizer = optim.AdamW(get_trainable_params(model), lr=cfg.head_lr, weight_decay=cfg.weight_decay)
    scheduler = None
    scaler = GradScaler(enabled=(device.type == 'cuda'))

    total_epochs = cfg.warmup_epochs + cfg.finetune_epochs
    best_auc = -1.0
    best_path = Path(cfg.output_dir) / f'{model_name}_best.pt'
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    epochs_no_improve = 0

    for epoch in range(1, total_epochs + 1):
        # Phase transition
        if epoch == cfg.warmup_epochs + 1:
            partial_unfreeze(model, model_name)
            optimizer = optim.AdamW(get_trainable_params(model), lr=cfg.ft_lr, weight_decay=cfg.weight_decay)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.finetune_epochs,
                eta_min=cfg.eta_min,
            )

        use_mixup = epoch > cfg.warmup_epochs
        model.train()
        train_losses = []

        for x, y, _ in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=(device.type == 'cuda')):
                if use_mixup:
                    x_mixed, y_a, y_b, lam = mixup_data(x, y, alpha=cfg.mixup_alpha)
                    logits = model(x_mixed).squeeze(1)
                    loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
                else:
                    logits = model(x).squeeze(1)
                    loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(loss.item())

        if scheduler is not None:
            scheduler.step()

        train_loss = float(np.mean(train_losses)) if train_losses else 0.0
        val_out = evaluate(model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_out['loss'])
        history['val_auc'].append(val_out['auc'])

        print(
            f'Epoch {epoch:02d}/{total_epochs} | '
            f'train_loss={train_loss:.4f} | val_loss={val_out["loss"]:.4f} | val_AUC={val_out["auc"]:.4f}'
        )

        if val_out['auc'] > best_auc:
            best_auc = val_out['auc']
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg.early_stopping_patience:
            print(f'Early stopping on epoch {epoch} (patience={cfg.early_stopping_patience}).')
            break

    return {
        'model_name': model_name,
        'best_path': str(best_path),
        'best_val_auc': float(best_auc),
        'history': history,
    }

## 5. Train All Models

In [ ]:
all_results = {}

for name in model_names:
    result = train_one_model(name)
    all_results[name] = result

print('\nSaved checkpoints:')
for name in model_names:
    print(f'- {name}: {all_results[name]["best_path"]}')

## 6. Model Selection & Threshold Tuning

In [ ]:
def binary_metrics_from_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'AUC-ROC': float(roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5),
        'F1': float(f1_score(y_true, y_pred, average='binary', zero_division=0)),
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'Recall': float(recall_score(y_true, y_pred, zero_division=0)),
    }


def load_best_model(name: str, ckpt_path: str):
    model = build_model(name).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model


comparison_rows = []
val_outputs_by_model = {}

for name in model_names:
    model = load_best_model(name, all_results[name]['best_path'])
    val_out = evaluate(model, val_loader, device)
    val_outputs_by_model[name] = val_out

    m = binary_metrics_from_probs(val_out['labels'], val_out['probs'], threshold=0.5)
    comparison_rows.append({'model': name, **m})

comparison_df = pd.DataFrame(comparison_rows).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
print('Validation model comparison (originals only):')
print(comparison_df)

best_model_name = comparison_df.loc[0, 'model']
best_val_auc = comparison_df.loc[0, 'AUC-ROC']
print(f'Selected model: {best_model_name} with val AUC = {best_val_auc:.4f}')

best_model = load_best_model(best_model_name, all_results[best_model_name]['best_path'])
best_val_out = val_outputs_by_model[best_model_name]

# Threshold tuning on validation set
thresholds = np.round(np.arange(0.30, 0.701, 0.01), 2)
threshold_scores = []
for t in thresholds:
    y_pred = (best_val_out['probs'] >= t).astype(int)
    macro_f1 = f1_score(best_val_out['labels'], y_pred, average='macro', zero_division=0)
    threshold_scores.append((float(t), float(macro_f1)))

best_threshold, best_macro_f1 = max(threshold_scores, key=lambda x: x[1])
print(f'Optimal threshold (val, max macro F1): {best_threshold:.2f} (macro F1={best_macro_f1:.4f})')

## 7. Final Evaluation on Test Set

In [ ]:
test_out = evaluate(best_model, test_loader, device)
test_metrics = binary_metrics_from_probs(test_out['labels'], test_out['probs'], threshold=best_threshold)

test_summary = pd.DataFrame([test_metrics])
print('Test metrics (selected best model, originals only):')
print(test_summary.to_string(index=False))

## 8. Visualisations & Export

In [ ]:
# 1) training_curves.png (3x3: rows=model, cols=train_loss/val_loss/val_auc)
fig, axes = plt.subplots(len(model_names), 3, figsize=(16, 4 * len(model_names)))
if len(model_names) == 1:
    axes = np.array([axes])

for r, name in enumerate(model_names):
    hist = all_results[name]['history']
    epochs = np.arange(1, len(hist['train_loss']) + 1)

    axes[r, 0].plot(epochs, hist['train_loss'], label='train_loss', color='tab:blue')
    axes[r, 0].set_title(f'{name} - Train Loss')
    axes[r, 1].plot(epochs, hist['val_loss'], label='val_loss', color='tab:orange')
    axes[r, 1].set_title(f'{name} - Val Loss')
    axes[r, 2].plot(epochs, hist['val_auc'], label='val_auc', color='tab:green')
    axes[r, 2].set_title(f'{name} - Val AUC')

    for c in range(3):
        axes[r, c].set_xlabel('Epoch')
        axes[r, c].grid(alpha=0.25)

        if name == best_model_name:
            for spine in axes[r, c].spines.values():
                spine.set_edgecolor('crimson')
                spine.set_linewidth(2.5)

plt.tight_layout()
training_curves_path = Path(cfg.output_dir) / 'training_curves.png'
plt.savefig(training_curves_path, dpi=200, bbox_inches='tight')
plt.show()

# 2) model_comparison.png (AUC, F1, Accuracy)
plot_df = comparison_df.copy()
metrics_to_plot = ['AUC-ROC', 'F1', 'Accuracy']
x = np.arange(len(plot_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, m in enumerate(metrics_to_plot):
    vals = plot_df[m].values
    colors = ['tab:gray'] * len(vals)
    winner_idx = int(np.where(plot_df['model'].values == best_model_name)[0][0])
    colors[winner_idx] = 'crimson'
    ax.bar(x + (i - 1) * width, vals, width=width, label=m, alpha=0.85, color=colors)

ax.set_xticks(x)
ax.set_xticklabels(plot_df['model'].tolist(), rotation=0)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison on Validation Set')
ax.legend()
ax.grid(axis='y', alpha=0.25)

model_cmp_path = Path(cfg.output_dir) / 'model_comparison.png'
plt.tight_layout()
plt.savefig(model_cmp_path, dpi=200, bbox_inches='tight')
plt.show()

# 3) confusion_matrix.png
test_pred = (test_out['probs'] >= best_threshold).astype(int)
cm = confusion_matrix(test_out['labels'].astype(int), test_pred.astype(int))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['nondiabetes', 'diabetes'],
            yticklabels=['nondiabetes', 'diabetes'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('True')
cm_path = Path(cfg.output_dir) / 'confusion_matrix.png'
plt.tight_layout()
plt.savefig(cm_path, dpi=200, bbox_inches='tight')
plt.show()

# 4) gradcam_samples.png (6 samples: 3 diabetes + 3 nondiabetes, originals only)
by_class_paths = {0: [], 1: []}
for p, y in test_ds.samples:
    if '__aug_' in p.name.lower():
        continue
    by_class_paths[int(y)].append(str(p))

selected_paths = by_class_paths[1][:3] + by_class_paths[0][:3]

cam = GradCAM(model=best_model, target_layers=get_gradcam_target_layer(best_model, best_model_name))

fig, axes = plt.subplots(len(selected_paths), 2, figsize=(10, 4 * len(selected_paths)))
if len(selected_paths) == 1:
    axes = np.array([axes])

for i, img_path in enumerate(selected_paths):
    pil = Image.open(img_path).convert('RGB').resize((cfg.img_size, cfg.img_size))
    raw_np = np.array(pil).astype(np.float32) / 255.0

    inp = val_test_tfms(pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logit = best_model(inp).squeeze(1)
        prob = torch.sigmoid(logit).item()

    grayscale_cam = cam(input_tensor=inp)[0]
    overlay = show_cam_on_image(raw_np, grayscale_cam, use_rgb=True)

    true_label = Path(img_path).parent.name

    axes[i, 0].imshow(raw_np)
    axes[i, 0].set_title(f'Raw | true={true_label}')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title(f'GradCAM | p(diabetes)={prob:.3f}')
    axes[i, 1].axis('off')

gradcam_path = Path(cfg.output_dir) / 'gradcam_samples.png'
plt.tight_layout()
plt.savefig(gradcam_path, dpi=200, bbox_inches='tight')
plt.show()

# 5) submission.csv (test originals only)
submission_df = pd.DataFrame({
    'filepath': test_out['paths'],
    'true_label': test_out['labels'].astype(int),
    'predicted_prob': test_out['probs'],
})
submission_df['predicted_label'] = (submission_df['predicted_prob'] >= best_threshold).astype(int)
submission_path = Path(cfg.output_dir) / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print('\nSaved artifacts:')
print('-', training_curves_path)
print('-', model_cmp_path)
print('-', cm_path)
print('-', gradcam_path)
print('-', submission_path)

## 9. Evaluation on External Dataset

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print('\n===== Evaluating on External Dataset =====')
# Use the best model found during training
external_out = evaluate(best_model, external_loader, device)
external_metrics = binary_metrics_from_probs(external_out['labels'], external_out['probs'], threshold=best_threshold)

external_summary = pd.DataFrame([external_metrics])
print('External Dataset Metrics:')
print(external_summary.to_string(index=False))

# 1) Generate Confusion Matrix
ext_pred = (external_out['probs'] >= best_threshold).astype(int)
cm_ext = confusion_matrix(external_out['labels'].astype(int), ext_pred.astype(int))

plt.figure(figsize=(6, 5))
sns.heatmap(cm_ext, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['nondiabetes', 'diabetes'],
            yticklabels=['nondiabetes', 'diabetes'])
plt.title(f'Confusion Matrix (External) - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('True')
cm_ext_path = Path(cfg.output_dir) / 'confusion_matrix_external.png'
plt.tight_layout()
plt.savefig(cm_ext_path, dpi=200, bbox_inches='tight')
plt.show()

# 2) GradCAM Visualizations for External Samples
by_class_paths_ext = {0: [], 1: []}
for p, y in external_ds.samples:
    by_class_paths_ext[int(y)].append(str(p))

selected_paths_ext = by_class_paths_ext[1][:3] + by_class_paths_ext[0][:3]

if len(selected_paths_ext) > 0:
    cam = GradCAM(model=best_model, target_layers=get_gradcam_target_layer(best_model, best_model_name))
    fig, axes = plt.subplots(len(selected_paths_ext), 2, figsize=(10, 4 * len(selected_paths_ext)))
    if len(selected_paths_ext) == 1: axes = np.array([axes])

    for i, img_path in enumerate(selected_paths_ext):
        pil = Image.open(img_path).convert('RGB').resize((cfg.img_size, cfg.img_size))
        raw_np = np.array(pil).astype(np.float32) / 255.0
        inp = val_test_tfms(pil).unsqueeze(0).to(device)
        
        with torch.no_grad():
            logit = best_model(inp).squeeze(1)
            prob = torch.sigmoid(logit).item()

        grayscale_cam = cam(input_tensor=inp)[0]
        overlay = show_cam_on_image(raw_np, grayscale_cam, use_rgb=True)
        true_label = Path(img_path).parent.name

        axes[i, 0].imshow(raw_np)
        axes[i, 0].set_title(f'Raw | true={true_label}')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f'GradCAM | p(diabetes)={prob:.3f}')
        axes[i, 1].axis('off')

    gradcam_ext_path = Path(cfg.output_dir) / 'gradcam_samples_external.png'
    plt.tight_layout()
    plt.savefig(gradcam_ext_path, dpi=200, bbox_inches='tight')
    plt.show()

# 3) Save results to CSV
external_df = pd.DataFrame({
    'filepath': external_out['paths'],
    'true_label': external_out['labels'].astype(int),
    'predicted_prob': external_out['probs'],
})
external_df['predicted_label'] = (external_df['predicted_prob'] >= best_threshold).astype(int)
external_path = Path(cfg.output_dir) / 'external_submission.csv'
external_df.to_csv(external_path, index=False)

print('\nSaved external artifacts:')
print('-', cm_ext_path)
print('-', external_path)